Name: Rushikesh Anil Khairnar  
Student: 5115301  
Roll NO: 21  
College: B. K. Birla College

# **Practical 8**
**1.Write a PySpark program to read a CSV dataset and display its schema.**



In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, round
import os

# ==========================================
# 0. SETUP: Initialize Spark and Mock Data
# ==========================================
# Initialize a SparkSession
spark = SparkSession.builder \
    .appName("PySpark_Data_Operations") \
    .master("local[*]") \
    .getOrCreate()

# Create mock sales data (contains a duplicate for transaction_id 3)
with open("sales.csv", "w") as f:
    f.write("transaction_id,product_id,quantity,sale_amount\n")
    f.write("1,101,2,50.0\n")
    f.write("2,102,1,120.0\n")
    f.write("3,101,4,100.0\n")
    f.write("3,101,4,100.0\n")  # Duplicate row
    f.write("4,103,1,30.0\n")
    f.write("5,101,0,0.0\n")    # Invalid quantity to filter out

# Create mock products data
with open("products.csv", "w") as f:
    f.write("product_id,product_name,category\n")
    f.write("101,Wireless Mouse,Electronics\n")
    f.write("102,Mechanical Keyboard,Electronics\n")
    f.write("103,Coffee Mug,Home & Kitchen\n")

# ==========================================
# 1. READ CSV & DISPLAY SCHEMA
# ==========================================
print("\n--- 1. Reading Data and Displaying Schema ---")
# Read CSV files into DataFrames with schema inference
sales_df = spark.read.csv("sales.csv", header=True, inferSchema=True)
products_df = spark.read.csv("products.csv", header=True, inferSchema=True)

# Display the schema of the sales DataFrame
sales_df.printSchema()


--- 1. Reading Data and Displaying Schema ---
root
 |-- transaction_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- sale_amount: double (nullable = true)



**2.Implement a PySpark program to remove duplicate records.**


In [2]:
# 3. REMOVE DUPLICATE RECORDS
# ==========================================
print("\n--- 2. Removing Duplicates ---")
print(f"Original row count: {sales_df.count()}")

# Drop duplicate rows across all columns
clean_sales_df = sales_df.dropDuplicates()

print(f"Row count after dropping duplicates: {clean_sales_df.count()}")
clean_sales_df.show()


--- 2. Removing Duplicates ---
Original row count: 6
Row count after dropping duplicates: 5
+--------------+----------+--------+-----------+
|transaction_id|product_id|quantity|sale_amount|
+--------------+----------+--------+-----------+
|             1|       101|       2|       50.0|
|             4|       103|       1|       30.0|
|             2|       102|       1|      120.0|
|             3|       101|       4|      100.0|
|             5|       101|       0|        0.0|
+--------------+----------+--------+-----------+



**3.Perform filtering, grouping, and aggregation operations using PySpark DataFrames.**


In [3]:
# ==========================================
# 2. FILTERING
# ==========================================
print("\n--- 3. Filtering Operations ---")
# Filter out records where quantity is 0 or less
filtered_sales_df = clean_sales_df.filter(col("quantity") > 0)
filtered_sales_df.show()


--- 3. Filtering Operations ---
+--------------+----------+--------+-----------+
|transaction_id|product_id|quantity|sale_amount|
+--------------+----------+--------+-----------+
|             1|       101|       2|       50.0|
|             4|       103|       1|       30.0|
|             2|       102|       1|      120.0|
|             3|       101|       4|      100.0|
+--------------+----------+--------+-----------+



**4.Perform a join between two datasets using PySpark.**


In [4]:
# 4. PERFORM A JOIN BETWEEN TWO DATASETS
# ==========================================
print("\n--- 4. Joining DataFrames ---")
# Perform an inner join on 'product_id'
joined_df = filtered_sales_df.join(products_df, on="product_id", how="inner")
joined_df.show()


--- 4. Joining DataFrames ---
+----------+--------------+--------+-----------+-------------------+--------------+
|product_id|transaction_id|quantity|sale_amount|       product_name|      category|
+----------+--------------+--------+-----------+-------------------+--------------+
|       101|             1|       2|       50.0|     Wireless Mouse|   Electronics|
|       103|             4|       1|       30.0|         Coffee Mug|Home & Kitchen|
|       102|             2|       1|      120.0|Mechanical Keyboard|   Electronics|
|       101|             3|       4|      100.0|     Wireless Mouse|   Electronics|
+----------+--------------+--------+-----------+-------------------+--------------+



**5.Write a PySpark program to calculate average sales by product category.**

In [5]:
# 5. GROUPING & AGGREGATION (Avg Sales by Category)
# ==========================================
print("\n--- 5. Average Sales by Product Category ---")
# Group by category, calculate the average sale amount, and round to 2 decimal places
avg_sales_df = joined_df.groupBy("category") \
    .agg(
        round(avg("sale_amount"), 2).alias("average_sales")
    )

avg_sales_df.show()

# ==========================================
# CLEANUP
# ==========================================
# Stop the Spark session and remove mock files
spark.stop()
os.remove("sales.csv")
os.remove("products.csv")


--- 5. Average Sales by Product Category ---
+--------------+-------------+
|      category|average_sales|
+--------------+-------------+
|Home & Kitchen|         30.0|
|   Electronics|         90.0|
+--------------+-------------+

